In [6]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Reload the saved sources.
REFRAME_DIR = (
    Path("data")
    / "interim"
    / "chronological_split_70_15_15"
    / "training_reframe"
)

model_baskets = pd.read_csv(
    REFRAME_DIR / "known_baskets_all_splits.csv"
)

examples_with_targets = pd.read_csv(
    REFRAME_DIR / "examples_with_targets.csv"
)

# Ensure every basket has a unique identifier.
assert model_baskets["block_id"].is_unique

# The largest known basket contains six items.
MAX_ITEMS_PER_BASKET = int(
    model_baskets["known_item_count"].max()
)

print("Maximum items in one basket:", MAX_ITEMS_PER_BASKET)

# Give every basket one numeric row position.
model_baskets = (
    model_baskets
    .sort_values(
        ["visitorid", "timestamp"],
        kind="stable",
    )
    .reset_index(drop=True)
)

model_baskets["basket_row_index"] = np.arange(
    len(model_baskets),
    dtype=np.int64,
)

# Matrix shape:
# rows = baskets
# columns = item positions inside each basket
basket_item_matrix = np.zeros(
    (
        len(model_baskets),
        MAX_ITEMS_PER_BASKET,
    ),
    dtype=np.int64,
)

for row_index, item_json in enumerate(
    model_baskets["item_indices"]
):
    item_list = json.loads(item_json)

    assert 1 <= len(item_list) <= MAX_ITEMS_PER_BASKET

    basket_item_matrix[
        row_index,
        :len(item_list),
    ] = item_list

# Map each example target block to its basket matrix row.
block_to_row = (
    model_baskets
    .set_index("block_id")["basket_row_index"]
)

mapped_target_rows = (
    examples_with_targets["target_block_id"]
    .map(block_to_row)
)

# Check for missing mappings before converting to integers.
assert mapped_target_rows.notna().all(), (
    "Some target_block_id values were not found in model_baskets."
)

examples_with_targets["target_basket_row"] = (
    mapped_target_rows.astype("int64")
)

# Check that each target row matches its visitor sequence position.
visitor_first_row = (
    model_baskets
    .groupby("visitorid")["basket_row_index"]
    .min()
)

expected_target_rows = (
    examples_with_targets["visitorid"]
    .map(visitor_first_row)
    + examples_with_targets["target_position"]
)

assert expected_target_rows.notna().all(), (
    "Some visitor IDs were not found in model_baskets."
)

assert examples_with_targets[
    "target_basket_row"
].eq(expected_target_rows).all(), (
    "Target basket rows do not match visitor sequence positions."
)

print("✅ Basket matrix shape:", basket_item_matrix.shape)
print("✅ Example targets linked to basket rows.")

print("\nFirst basket matrix row:")
print(basket_item_matrix[0])

display(
    examples_with_targets[
        [
            "visitorid",
            "split",
            "history_length",
            "target_basket_row",
            "target_item_indices",
        ]
    ].head()
)

Maximum items in one basket: 6
✅ Basket matrix shape: (2680431, 6)
✅ Example targets linked to basket rows.

First basket matrix row:
[123222      0      0      0      0      0]


,visitorid,split,history_length,target_basket_row,target_item_indices
0,0,test,1,1,[154167]
1,0,test,2,2,[28938]
2,2,validation,1,5,[140024]
3,2,validation,2,6,[112081]
4,2,validation,3,7,[93296]


In [2]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    properties = torch.cuda.get_device_properties(0)

    total_gb = properties.total_memory / (1024 ** 3)
    allocated_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)
    free_gb = total_gb - reserved_gb

    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total GPU memory:     {total_gb:.2f} GB")
    print(f"PyTorch allocated:     {allocated_gb:.2f} GB")
    print(f"PyTorch reserved:      {reserved_gb:.2f} GB")
    print(f"Estimated free memory: {free_gb:.2f} GB")
else:
    print("❌ PyTorch cannot see a CUDA GPU.")

GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Total GPU memory:     8.00 GB
PyTorch allocated:     0.00 GB
PyTorch reserved:      0.00 GB
Estimated free memory: 8.00 GB


In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
NVIDIA GeForce RTX 4070 Laptop GPU


In [7]:
import json
import numpy as np

model_baskets = pd.read_csv(
    REFRAME_DIR / "known_baskets_all_splits.csv"
)

examples_with_targets = pd.read_csv(
    REFRAME_DIR / "examples_with_targets.csv"
)

In [9]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Efficient fixed shapes for your RTX 4070 (8 GB).
MAX_HISTORY_BASKETS = 20
BATCH_SIZE = 128

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "CUDA GPU was not detected."

# ------------------------------------------------------------
# Recreate the temporary target block → basket matrix row link.
# This is needed again after a Jupyter kernel restart.
# ------------------------------------------------------------
model_baskets = (
    model_baskets
    .sort_values(["visitorid", "timestamp"], kind="stable")
    .reset_index(drop=True)
)

model_baskets["basket_row_index"] = np.arange(
    len(model_baskets),
    dtype=np.int64,
)

block_to_row = model_baskets.set_index(
    "block_id"
)["basket_row_index"]

examples_with_targets["target_basket_row"] = (
    examples_with_targets["target_block_id"]
    .map(block_to_row)
    .astype("int64")
)

assert examples_with_targets["target_basket_row"].notna().all()
assert examples_with_targets["target_basket_row"].between(
    0,
    len(basket_item_matrix) - 1,
).all()

# Read the matrix shape instead of depending on an older variable.
MAX_ITEMS_PER_BASKET = basket_item_matrix.shape[1]

print("✅ Target basket rows recreated.")
print("✅ Maximum items per basket:", MAX_ITEMS_PER_BASKET)


class NextBasketDataset(Dataset):
    """
    One example:
        history_items = up to 20 earlier baskets
        target_items  = items in the immediately next basket
    """

    def __init__(self, examples, basket_matrix):
        self.basket_matrix = basket_matrix

        # Keep only the small numeric arrays needed per example.
        self.target_rows = examples[
            "target_basket_row"
        ].to_numpy(dtype=np.int64)

        self.history_lengths = examples[
            "history_length"
        ].to_numpy(dtype=np.int64)

        assert (self.history_lengths >= 1).all()
        assert (self.history_lengths <= MAX_HISTORY_BASKETS).all()
        assert (self.target_rows >= self.history_lengths).all()

    def __len__(self):
        return len(self.target_rows)

    def __getitem__(self, index):
        target_row = self.target_rows[index]
        history_length = self.history_lengths[index]

        # Fixed input shape: [20 baskets, 6 item slots].
        # Zero means padding, never a real item.
        history = np.zeros(
            (
                MAX_HISTORY_BASKETS,
                MAX_ITEMS_PER_BASKET,
            ),
            dtype=np.int64,
        )

        history_start = target_row - history_length

        # Place real history on the right.
        history[-history_length:] = self.basket_matrix[
            history_start:target_row
        ]

        # Full next basket: one to six correct target items.
        target_items = self.basket_matrix[target_row].copy()

        return {
            "history_items": torch.from_numpy(history),
            "history_length": torch.tensor(
                history_length,
                dtype=torch.long,
            ),
            "target_items": torch.from_numpy(target_items),
        }


# Keep the existing chronological split.
train_examples = examples_with_targets.loc[
    examples_with_targets["split"].eq("train")
].reset_index(drop=True)

validation_examples = examples_with_targets.loc[
    examples_with_targets["split"].eq("validation")
].reset_index(drop=True)

test_examples = examples_with_targets.loc[
    examples_with_targets["split"].eq("test")
].reset_index(drop=True)

train_dataset = NextBasketDataset(
    train_examples,
    basket_item_matrix,
)

validation_dataset = NextBasketDataset(
    validation_examples,
    basket_item_matrix,
)

test_dataset = NextBasketDataset(
    test_examples,
    basket_item_matrix,
)

# Training examples are shuffled between batches only.
# History order inside every example stays chronological.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

print("\n✅ Train examples:", len(train_dataset))
print("✅ Validation examples:", len(validation_dataset))
print("✅ Test examples:", len(test_dataset))

# Inspect one batch without training a model.
sample_batch = next(iter(train_loader))

print("\nBatch shapes:")
for name, tensor in sample_batch.items():
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

# Test a non-blocking transfer to the GPU.
gpu_history = sample_batch["history_items"].to(
    device,
    non_blocking=True,
)

print("\n✅ GPU transfer worked:", gpu_history.device)

✅ Target basket rows recreated.
✅ Maximum items per basket: 6

✅ Train examples: 948161
✅ Validation examples: 182936
✅ Test examples: 173786

Batch shapes:
history_items: shape=(128, 20, 6), dtype=torch.int64
history_length: shape=(128,), dtype=torch.int64
target_items: shape=(128, 6), dtype=torch.int64

✅ GPU transfer worked: cuda:0


In [10]:
import torch.nn as nn

# Item index 0 is padding.
# Real training items start at 1.
NUM_TRAIN_ITEMS = int(basket_item_matrix.max())

# Small, efficient starting size for the 8 GB RTX 4070.
EMBEDDING_DIM = 64

print("Real training items:", NUM_TRAIN_ITEMS)
print("Embedding dimension:", EMBEDDING_DIM)


class BasketEncoder(nn.Module):
    """
    Converts:

        [batch, history baskets, item slots]
        [128, 20, 6]

    into:

        [batch, history baskets, basket embedding]
        [128, 20, 64]
    """

    def __init__(self, num_items, embedding_dim):
        super().__init__()

        self.item_embedding = nn.Embedding(
            num_embeddings=num_items + 1,
            embedding_dim=embedding_dim,
            padding_idx=0,  # item ID 0 is padding only
        )

    def forward(self, basket_item_ids):
        """
        basket_item_ids shape:
            [batch_size, max_history_baskets, max_items_per_basket]
        """

        # True for real item IDs; False for padding zeros.
        item_mask = basket_item_ids.ne(0)

        # Shape:
        # [batch, 20, 6, 64]
        item_vectors = self.item_embedding(basket_item_ids)

        # Add item vectors inside each basket.
        item_vector_sum = (
            item_vectors
            * item_mask.unsqueeze(-1)
        ).sum(dim=2)

        # Count real items in every basket.
        # clamp(min=1) prevents division by zero for padded baskets.
        item_count = item_mask.sum(
            dim=2,
            keepdim=True,
        ).clamp(min=1)

        # Mean-pool real item vectors into one basket vector.
        # Shape: [batch, 20, 64]
        basket_vectors = item_vector_sum / item_count

        # True only for history positions containing at least one real item.
        # Shape: [batch, 20]
        basket_mask = item_mask.any(dim=2)

        return basket_vectors, basket_mask


# Create the basket encoder on the GPU.
basket_encoder = BasketEncoder(
    num_items=NUM_TRAIN_ITEMS,
    embedding_dim=EMBEDDING_DIM,
).to(device)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in basket_encoder.parameters()):,}",
)

# Use the batch already created by the DataLoader.
history_items_gpu = sample_batch["history_items"].to(
    device,
    non_blocking=True,
)

# Test one forward pass.
# This is not training; it only checks the tensor flow.
with torch.no_grad():
    basket_vectors, basket_mask = basket_encoder(
        history_items_gpu
    )

print("\n✅ Basket encoder output:")
print("basket_vectors shape:", tuple(basket_vectors.shape))
print("basket_mask shape:", tuple(basket_mask.shape))

# Safety check: padded history positions must be marked False.
assert basket_mask.dtype == torch.bool
assert basket_vectors.shape == (
    BATCH_SIZE,
    MAX_HISTORY_BASKETS,
    EMBEDDING_DIM,
)

print("\nFirst example's 20 history-position mask:")
print(basket_mask[0].cpu().numpy())

Real training items: 200978
Embedding dimension: 64
Trainable parameters: 12,862,656

✅ Basket encoder output:
basket_vectors shape: (128, 20, 64)
basket_mask shape: (128, 20)

First example's 20 history-position mask:
[False False False False False False False False False False False False
 False False False False  True  True  True  True]


In [11]:
import torch.nn as nn

# Fast matrix multiplication on RTX 4070.
# It uses TF32 where safe and keeps model quality practical.
torch.set_float32_matmul_precision("high")

# Small starting configuration for an 8 GB GPU.
NUM_TRANSFORMER_LAYERS = 2
NUM_ATTENTION_HEADS = 2
FEED_FORWARD_DIM = 128
DROPOUT = 0.10


class BasketSASRec(nn.Module):
    """
    Basket-aware SASRec backbone.

    Flow:
        item IDs
        → mean-pooled basket vectors
        → positional information
        → causal Transformer
        → latest-history vector for next-item prediction

    The causal mask ensures a basket can only attend to itself
    and earlier baskets, never future baskets.
    """

    def __init__(
        self,
        basket_encoder,
        max_history_baskets,
        embedding_dim,
        num_layers,
        num_heads,
        feed_forward_dim,
        dropout,
    ):
        super().__init__()

        # Reuse the already-created item/basket encoder.
        self.basket_encoder = basket_encoder
        self.max_history_baskets = max_history_baskets

        # Position 0 to 19: gives the Transformer sequence location.
        self.position_embedding = nn.Embedding(
            num_embeddings=max_history_baskets,
            embedding_dim=embedding_dim,
        )

        self.input_dropout = nn.Dropout(dropout)

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=feed_forward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,  # [batch, sequence, embedding]
            norm_first=False,
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )

        self.output_norm = nn.LayerNorm(embedding_dim)

        # True means "do not look here".
        # Upper triangle blocks attention toward future positions.
        causal_mask = torch.triu(
            torch.ones(
                max_history_baskets,
                max_history_baskets,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

        # Saved with the model and moved automatically to GPU.
        self.register_buffer(
            "causal_mask",
            causal_mask,
            persistent=False,
        )

    def forward(self, history_items):
        """
        history_items shape:
            [batch, 20, 6]

        Returns:
            latest_context: [batch, 64]
            encoded_history: [batch, 20, 64]
            basket_mask: [batch, 20]
        """

        # Step 1: items inside each basket become one basket vector.
        basket_vectors, basket_mask = self.basket_encoder(
            history_items
        )

        batch_size, sequence_length, _ = basket_vectors.shape

        assert sequence_length == self.max_history_baskets

        # Step 2: add position embeddings.
        positions = torch.arange(
            sequence_length,
            device=history_items.device,
        )

        position_vectors = self.position_embedding(positions)

        transformer_input = basket_vectors + position_vectors.unsqueeze(0)

        # Keep padded history positions as zero vectors.
        transformer_input = transformer_input * (
            basket_mask.unsqueeze(-1)
        )

        transformer_input = self.input_dropout(transformer_input)

        # Step 3: causal Transformer.
        # src_key_padding_mask=True means "ignore this padded position".
        encoded_history = self.transformer(
            transformer_input,
            mask=self.causal_mask,
            src_key_padding_mask=~basket_mask,
        )

        encoded_history = self.output_norm(encoded_history)

        # Ensure padding positions remain zero after Transformer operations.
        encoded_history = encoded_history * (
            basket_mask.unsqueeze(-1)
        )

        # Find the most recent real basket for each example.
        sequence_positions = torch.arange(
            sequence_length,
            device=history_items.device,
        ).unsqueeze(0)

        last_real_position = torch.where(
            basket_mask,
            sequence_positions,
            torch.full_like(sequence_positions, -1),
        ).max(dim=1).values

        assert (last_real_position >= 0).all()

        latest_context = encoded_history[
            torch.arange(batch_size, device=history_items.device),
            last_real_position,
        ]

        return latest_context, encoded_history, basket_mask


# The existing basket_encoder is reused: no duplicate item-embedding table.
sasrec_model = BasketSASRec(
    basket_encoder=basket_encoder,
    max_history_baskets=MAX_HISTORY_BASKETS,
    embedding_dim=EMBEDDING_DIM,
    num_layers=NUM_TRANSFORMER_LAYERS,
    num_heads=NUM_ATTENTION_HEADS,
    feed_forward_dim=FEED_FORWARD_DIM,
    dropout=DROPOUT,
).to(device)

print(
    "Total trainable parameters:",
    f"{sum(p.numel() for p in sasrec_model.parameters()):,}",
)

# Test one forward pass using BF16 mixed precision.
# This reduces GPU memory pressure and will be used during training later.
history_items_gpu = sample_batch["history_items"].to(
    device,
    non_blocking=True,
)

sasrec_model.eval()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
):
    latest_context, encoded_history, output_mask = sasrec_model(
        history_items_gpu
    )

print("\n✅ SASRec backbone output:")
print("latest_context:", tuple(latest_context.shape))
print("encoded_history:", tuple(encoded_history.shape))
print("output_mask:", tuple(output_mask.shape))

assert latest_context.shape == (
    BATCH_SIZE,
    EMBEDDING_DIM,
)

assert encoded_history.shape == (
    BATCH_SIZE,
    MAX_HISTORY_BASKETS,
    EMBEDDING_DIM,
)

torch.cuda.synchronize()

print(
    "\nPyTorch GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB",
)
print(
    "PyTorch GPU reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB",
)

Total trainable parameters: 12,931,008

✅ SASRec backbone output:
latest_context: (128, 64)
encoded_history: (128, 20, 64)
output_mask: (128, 20)

PyTorch GPU allocated: 0.06 GB
PyTorch GPU reserved: 0.08 GB


In [12]:
import math
import torch.nn.functional as F

# Efficient training setting:
# each real target competes against 256 sampled wrong items.
NUM_NEGATIVE_ITEMS = 256


class TiedItemScorer(nn.Module):
    """
    Scores candidate items using the same item embeddings used
    to build baskets. This avoids a second 200k-item embedding table.
    """

    def __init__(self, item_embedding, embedding_dim):
        super().__init__()

        self.item_embedding = item_embedding
        self.score_scale = embedding_dim ** -0.5

    def forward(self, context_vectors, candidate_item_ids):
        """
        context_vectors:
            [positive target pairs, embedding_dim]

        candidate_item_ids:
            [positive target pairs, 1 + negative items]

        Returns:
            one score per candidate item.
        """

        candidate_vectors = self.item_embedding(
            candidate_item_ids
        )

        scores = torch.sum(
            context_vectors.unsqueeze(1) * candidate_vectors,
            dim=-1,
        )

        return scores * self.score_scale


class SampledMultiPositiveLoss(nn.Module):
    """
    Every item in the next basket is treated as correct.

    Example:
        next basket = [item_12, item_45]

    This creates two positive learning signals:
        history → item_12
        history → item_45

    For speed, each positive item is compared with sampled
    negative items from the training-only catalogue.
    """

    def __init__(self, scorer, num_items, num_negative_items):
        super().__init__()

        self.scorer = scorer
        self.num_items = num_items
        self.num_negative_items = num_negative_items

    def forward(self, context_vectors, target_items):
        """
        context_vectors: [batch, 64]
        target_items:    [batch, 6]

        Zero in target_items is only basket padding.
        """

        # Find all real target items in each next basket.
        positive_mask = target_items.ne(0)

        positive_count_per_example = positive_mask.sum(dim=1)

        assert (positive_count_per_example >= 1).all()

        # Flatten all positive targets.
        # Most baskets have one target; some have multiple.
        positive_item_ids = target_items[positive_mask]

        # Repeat the matching context for each positive target item.
        positive_contexts = torch.repeat_interleave(
            context_vectors,
            positive_count_per_example,
            dim=0,
        )

        # Remember which original basket each positive target came from.
        source_example_index = torch.repeat_interleave(
            torch.arange(
                target_items.size(0),
                device=target_items.device,
            ),
            positive_count_per_example,
        )

        # Sample wrong items only from the training catalogue:
        # valid IDs are 1 to NUM_TRAIN_ITEMS.
        negative_item_ids = torch.randint(
            low=1,
            high=self.num_items + 1,
            size=(
                len(positive_item_ids),
                self.num_negative_items,
            ),
            device=target_items.device,
        )

        # A sampled negative must not be any true item in that
        # example's complete next basket.
        forbidden_items = target_items[
            source_example_index
        ]

        collision_mask = (
            negative_item_ids.unsqueeze(-1)
            .eq(forbidden_items.unsqueeze(1))
            .any(dim=-1)
        )

        # Resample only rare collisions.
        while collision_mask.any():
            negative_item_ids[collision_mask] = torch.randint(
                low=1,
                high=self.num_items + 1,
                size=(int(collision_mask.sum().item()),),
                device=target_items.device,
            )

            collision_mask = (
                negative_item_ids.unsqueeze(-1)
                .eq(forbidden_items.unsqueeze(1))
                .any(dim=-1)
            )

        # Candidate at position 0 is always the correct item.
        candidate_item_ids = torch.cat(
            [
                positive_item_ids.unsqueeze(1),
                negative_item_ids,
            ],
            dim=1,
        )

        logits = self.scorer(
            positive_contexts,
            candidate_item_ids,
        )

        # Class 0 means the first candidate is the correct one.
        correct_class = torch.zeros(
            len(positive_item_ids),
            dtype=torch.long,
            device=target_items.device,
        )

        # Convert to float32 for numerically stable loss calculation.
        loss = F.cross_entropy(
            logits.float(),
            correct_class,
        )

        return loss, {
            "positive_target_pairs": int(
                len(positive_item_ids)
            ),
            "candidates_per_positive": int(
                candidate_item_ids.size(1)
            ),
        }


# The scorer reuses SASRec's existing item-embedding table.
item_scorer = TiedItemScorer(
    item_embedding=sasrec_model.basket_encoder.item_embedding,
    embedding_dim=EMBEDDING_DIM,
).to(device)

multi_positive_loss = SampledMultiPositiveLoss(
    scorer=item_scorer,
    num_items=NUM_TRAIN_ITEMS,
    num_negative_items=NUM_NEGATIVE_ITEMS,
).to(device)

# ------------------------------------------------------------
# Test one forward pass and loss calculation.
# This is still not an optimiser/training loop.
# ------------------------------------------------------------
history_items_gpu = sample_batch["history_items"].to(
    device,
    non_blocking=True,
)

target_items_gpu = sample_batch["target_items"].to(
    device,
    non_blocking=True,
)

sasrec_model.eval()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
):
    latest_context, _, _ = sasrec_model(history_items_gpu)

    test_loss, loss_info = multi_positive_loss(
        latest_context,
        target_items_gpu,
    )

assert torch.isfinite(test_loss)

print("✅ Sampled multi-positive loss:", test_loss.item())
print(
    "Positive target pairs in this batch:",
    loss_info["positive_target_pairs"],
)
print(
    "Candidates per positive target:",
    loss_info["candidates_per_positive"],
)

torch.cuda.synchronize()

print(
    "PyTorch GPU allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB",
)

✅ Sampled multi-positive loss: 4.872773170471191
Positive target pairs in this batch: 128
Candidates per positive target: 257
PyTorch GPU allocated: 0.06 GB


In [19]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 1. Reproducible experiment setup
# ------------------------------------------------------------
PROJECT_ROOT = Path(
    r"E:\ALL MCP Servers\Recommendation"
)

assert PROJECT_ROOT.exists(), "Project folder was not found."

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reinitialize model weights now that the seed is fixed.
# This makes the first real experiment reproducible.
def reset_trainable_module(module):
    if isinstance(module, nn.Embedding):
        module.reset_parameters()

    elif isinstance(module, nn.Linear):
        module.reset_parameters()

    elif isinstance(module, nn.LayerNorm):
        module.reset_parameters()

    elif isinstance(module, nn.MultiheadAttention):
        module._reset_parameters()


sasrec_model.apply(reset_trainable_module)

# ------------------------------------------------------------
# 2. Efficient training configuration
# ------------------------------------------------------------
TRAIN_BATCH_SIZE = 256
VALIDATION_BATCH_SIZE = 512

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
GRADIENT_CLIP_NORM = 1.0

# Bigger batches reduce the total number of training steps.
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

# Standard optimizer for Transformer models.
optimizer = torch.optim.AdamW(
    sasrec_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Folder for the best model and readable training history.
MODEL_DIR = PROJECT_ROOT / "models" / "sasrec_basket_v1"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "best_model.pt"
HISTORY_PATH = MODEL_DIR / "training_history.csv"

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Best model path:", BEST_MODEL_PATH)


# ------------------------------------------------------------
# 3. One complete training epoch
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, loss_function):
    model.train()

    weighted_loss_sum = 0.0
    positive_pair_count = 0

    epoch_start = time.perf_counter()

    for step, batch in enumerate(loader, start=1):
        history_items = batch["history_items"].to(
            device,
            non_blocking=True,
        )

        target_items = batch["target_items"].to(
            device,
            non_blocking=True,
        )

        # Frees old gradients rather than filling them with zeros.
        optimizer.zero_grad(set_to_none=True)

        # BF16 reduces GPU memory use on the RTX 4070.
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            latest_context, _, _ = model(history_items)

            loss, loss_info = loss_function(
                latest_context,
                target_items,
            )

        loss.backward()

        # Prevents unstable updates from unusually large gradients.
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP_NORM,
        )

        optimizer.step()

        pair_count = loss_info["positive_target_pairs"]

        weighted_loss_sum += loss.item() * pair_count
        positive_pair_count += pair_count

        if step % 500 == 0:
            running_loss = (
                weighted_loss_sum / positive_pair_count
            )

            print(
                f"  Step {step:,}/{len(loader):,} | "
                f"training loss: {running_loss:.4f}"
            )

    epoch_seconds = time.perf_counter() - epoch_start

    return (
        weighted_loss_sum / positive_pair_count,
        epoch_seconds,
    )


# ------------------------------------------------------------
# 4. Validation without gradients
# ------------------------------------------------------------
@torch.inference_mode()
def evaluate_validation_loss(model, loader, loss_function):
    model.eval()

    weighted_loss_sum = 0.0
    positive_pair_count = 0

    # device.index was None because device was "cuda".
    # current_device() correctly returns GPU number 0.
    gpu_number = torch.cuda.current_device()

    # Fixed seed = identical validation negatives every epoch.
    with torch.random.fork_rng(devices=[gpu_number]):
        torch.manual_seed(SEED + 10_000)
        torch.cuda.manual_seed_all(SEED + 10_000)

        for batch in loader:
            history_items = batch["history_items"].to(
                device,
                non_blocking=True,
            )

            target_items = batch["target_items"].to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                latest_context, _, _ = model(history_items)

                loss, loss_info = loss_function(
                    latest_context,
                    target_items,
                )

            pair_count = loss_info["positive_target_pairs"]

            weighted_loss_sum += loss.item() * pair_count
            positive_pair_count += pair_count

    return weighted_loss_sum / positive_pair_count


# ------------------------------------------------------------
# 5. Train, validate, save best model, early stop if needed
# ------------------------------------------------------------
best_validation_loss = float("inf")
epochs_without_improvement = 0
training_history = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for epoch in range(1, MAX_EPOCHS + 1):
    print(f"\n{'=' * 60}")
    print(f"Epoch {epoch}/{MAX_EPOCHS}")

    train_loss, epoch_seconds = train_one_epoch(
        sasrec_model,
        train_loader,
        optimizer,
        multi_positive_loss,
    )

    validation_loss = evaluate_validation_loss(
        sasrec_model,
        validation_loader,
        multi_positive_loss,
    )

    improved = validation_loss < best_validation_loss

    record = {
        "epoch": epoch,
        "train_sampled_loss": train_loss,
        "validation_sampled_loss": validation_loss,
        "epoch_seconds": epoch_seconds,
        "gpu_peak_allocated_gb": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        "improved": improved,
    }

    training_history.append(record)

    print(f"Train loss:      {train_loss:.4f}")
    print(f"Validation loss: {validation_loss:.4f}")
    print(f"Epoch time:      {epoch_seconds / 60:.2f} minutes")
    print(
        "Peak GPU use:    "
        f"{record['gpu_peak_allocated_gb']:.2f} GB"
    )

    if improved:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": sasrec_model.state_dict(),
                "epoch": epoch,
                "validation_sampled_loss": validation_loss,
                "config": {
                    "seed": SEED,
                    "embedding_dim": EMBEDDING_DIM,
                    "max_history_baskets": MAX_HISTORY_BASKETS,
                    "max_items_per_basket": MAX_ITEMS_PER_BASKET,
                    "num_transformer_layers": NUM_TRANSFORMER_LAYERS,
                    "num_attention_heads": NUM_ATTENTION_HEADS,
                    "feed_forward_dim": FEED_FORWARD_DIM,
                    "dropout": DROPOUT,
                    "training_batch_size": TRAIN_BATCH_SIZE,
                    "num_negative_items": NUM_NEGATIVE_ITEMS,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                },
            },
            BEST_MODEL_PATH,
        )

        print("✅ Best validation model saved.")

    else:
        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print("⏹️ Early stopping activated.")
            break


# ------------------------------------------------------------
# 6. Save readable experiment history
# ------------------------------------------------------------
training_history_df = pd.DataFrame(training_history)

training_history_df.to_csv(
    HISTORY_PATH,
    index=False,
)

display(training_history_df)

print("\n✅ Training run finished.")
print("Best validation loss:", round(best_validation_loss, 4))
print("Best checkpoint:", BEST_MODEL_PATH)
print("History file:", HISTORY_PATH)
def save_best_checkpoint(epoch, validation_loss):
    torch.save(
        {
            "optimizer_state_dict": optimizer.state_dict(),
            "model_state_dict": sasrec_model.state_dict(),
            "epoch": epoch,
            "validation_sampled_loss": validation_loss,
            "config": {
                "seed": SEED,
                "embedding_dim": EMBEDDING_DIM,
                "max_history_baskets": MAX_HISTORY_BASKETS,
                "max_items_per_basket": MAX_ITEMS_PER_BASKET,
                "num_transformer_layers": NUM_TRANSFORMER_LAYERS,
                "num_attention_heads": NUM_ATTENTION_HEADS,
                "feed_forward_dim": FEED_FORWARD_DIM,
                "dropout": DROPOUT,
                "training_batch_size": TRAIN_BATCH_SIZE,
                "num_negative_items": NUM_NEGATIVE_ITEMS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
            },
        },
        BEST_MODEL_PATH,
    )


# ------------------------------------------------------------
# Validate the already-trained Epoch 1 model.
# train_loss and epoch_seconds came from the completed Epoch 1.
# ------------------------------------------------------------
print("Validating completed Epoch 1...")

validation_loss = evaluate_validation_loss(
    sasrec_model,
    validation_loader,
    multi_positive_loss,
)

best_validation_loss = validation_loss
epochs_without_improvement = 0

epoch_1_record = {
    "epoch": 1,
    "train_sampled_loss": train_loss,
    "validation_sampled_loss": validation_loss,
    "epoch_seconds": epoch_seconds,
    "gpu_peak_allocated_gb": (
        torch.cuda.max_memory_allocated() / 1024**3
    ),
    "improved": True,
}

# Add Epoch 1 only once.
if not any(
    record["epoch"] == 1
    for record in training_history
):
    training_history.append(epoch_1_record)

save_best_checkpoint(
    epoch=1,
    validation_loss=validation_loss,
)

print(f"Epoch 1 train loss:      {train_loss:.4f}")
print(f"Epoch 1 validation loss: {validation_loss:.4f}")
print("✅ Epoch 1 best checkpoint saved.")


# ------------------------------------------------------------
# Continue from Epoch 2. Epoch 1 is not repeated.
# ------------------------------------------------------------
for epoch in range(2, MAX_EPOCHS + 1):
    print(f"\n{'=' * 60}")
    print(f"Epoch {epoch}/{MAX_EPOCHS}")

    train_loss, epoch_seconds = train_one_epoch(
        sasrec_model,
        train_loader,
        optimizer,
        multi_positive_loss,
    )

    validation_loss = evaluate_validation_loss(
        sasrec_model,
        validation_loader,
        multi_positive_loss,
    )

    improved = validation_loss < best_validation_loss

    record = {
        "epoch": epoch,
        "train_sampled_loss": train_loss,
        "validation_sampled_loss": validation_loss,
        "epoch_seconds": epoch_seconds,
        "gpu_peak_allocated_gb": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        "improved": improved,
    }

    training_history.append(record)

    print(f"Train loss:      {train_loss:.4f}")
    print(f"Validation loss: {validation_loss:.4f}")
    print(f"Epoch time:      {epoch_seconds / 60:.2f} minutes")
    print(
        "Peak GPU use:    "
        f"{record['gpu_peak_allocated_gb']:.2f} GB"
    )

    if improved:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0

        save_best_checkpoint(
            epoch=epoch,
            validation_loss=validation_loss,
        )

        print("✅ Best validation model saved.")

    else:
        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print("⏹️ Early stopping activated.")
            break


training_history_df = pd.DataFrame(training_history)

training_history_df.to_csv(
    HISTORY_PATH,
    index=False,
)

display(training_history_df)

print("\n✅ Training run finished.")
print("Best validation loss:", round(best_validation_loss, 4))
print("Best checkpoint:", BEST_MODEL_PATH)
print("History file:", HISTORY_PATH)

Training examples: 948161
Validation examples: 182936
Best model path: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\best_model.pt

Epoch 1/5
  Step 500/3,704 | training loss: 4.1774
  Step 1,000/3,704 | training loss: 4.0832
  Step 1,500/3,704 | training loss: 4.0029
  Step 2,000/3,704 | training loss: 3.9176
  Step 2,500/3,704 | training loss: 3.8274
  Step 3,000/3,704 | training loss: 3.7367
  Step 3,500/3,704 | training loss: 3.6485
Train loss:      3.6145
Validation loss: 3.0221
Epoch time:      1.77 minutes
Peak GPU use:    0.38 GB
✅ Best validation model saved.

Epoch 2/5
  Step 500/3,704 | training loss: 2.8222
  Step 1,000/3,704 | training loss: 2.7779
  Step 1,500/3,704 | training loss: 2.7438
  Step 2,000/3,704 | training loss: 2.7152
  Step 2,500/3,704 | training loss: 2.6925
  Step 3,000/3,704 | training loss: 2.6707
  Step 3,500/3,704 | training loss: 2.6529
Train loss:      2.6450
Validation loss: 2.7611
Epoch time:      1.33 minutes
Peak GPU use:    0.38 GB


,epoch,train_sampled_loss,validation_sampled_loss,epoch_seconds,gpu_peak_allocated_gb,improved
0,1,3.614515,3.022108,106.361554,0.38123,True
1,2,2.645007,2.761129,79.947715,0.38123,True
2,3,2.310318,2.670076,56.280238,0.38123,True
3,4,2.066467,2.596696,57.290741,0.38123,True
4,5,1.843023,2.538252,55.105036,0.38123,True



✅ Training run finished.
Best validation loss: 2.5383
Best checkpoint: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\best_model.pt
History file: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\training_history.csv
Validating completed Epoch 1...
Epoch 1 train loss:      1.8430
Epoch 1 validation loss: 2.5383
✅ Epoch 1 best checkpoint saved.

Epoch 2/5
  Step 500/3,704 | training loss: 1.6616
  Step 1,000/3,704 | training loss: 1.6611
  Step 1,500/3,704 | training loss: 1.6566
  Step 2,000/3,704 | training loss: 1.6512
  Step 2,500/3,704 | training loss: 1.6454


KeyboardInterrupt: 

In [17]:
import math
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Load the best validation checkpoint
# ------------------------------------------------------------
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

sasrec_model.eval()

print("Loaded best checkpoint from epoch:", best_checkpoint["epoch"])
print(
    "Best validation sampled loss:",
    round(best_checkpoint["validation_sampled_loss"], 4),
)


# ------------------------------------------------------------
# 2. Fixed candidate-ranking protocol
# ------------------------------------------------------------
# Each validation example gets:
# - all correct items from its next basket
# - 100 distinct wrong items from the training catalogue
#
# This is sampled-candidate evaluation, not full-catalogue ranking.
NUM_EVALUATION_NEGATIVES = 100
RANKING_K_VALUES = [10, 20]
EVALUATION_SEED = 20_042


def sample_distinct_negative_items(
    target_items,
    num_items,
    num_negative_items,
):
    """
    Draw distinct negatives from the training catalogue.

    Negatives cannot be:
    - item ID 0 (padding)
    - any correct target item in the same next basket
    - duplicates of another sampled negative
    """

    batch_size = target_items.size(0)

    negative_items = torch.randint(
        low=1,
        high=num_items + 1,
        size=(batch_size, num_negative_items),
        device=target_items.device,
    )

    while True:
        # Never sample a true next-basket item as a negative.
        target_collision = (
            negative_items.unsqueeze(-1)
            .eq(target_items.unsqueeze(1))
            .any(dim=-1)
        )

        # Never allow duplicate negative candidates in one example.
        duplicate_collision = torch.zeros_like(
            negative_items,
            dtype=torch.bool,
        )

        for position in range(1, num_negative_items):
            duplicate_collision[:, position] = (
                negative_items[:, position].unsqueeze(1)
                .eq(negative_items[:, :position])
                .any(dim=1)
            )

        invalid_negative = (
            target_collision | duplicate_collision
        )

        if not invalid_negative.any():
            break

        # Resample only invalid locations.
        negative_items[invalid_negative] = torch.randint(
            low=1,
            high=num_items + 1,
            size=(int(invalid_negative.sum().item()),),
            device=target_items.device,
        )

    return negative_items


def update_ranking_totals(
    scores,
    candidate_item_ids,
    target_items,
    metric_totals,
):
    """
    Updates multi-positive Recall@K and NDCG@K totals.

    All real next-basket items are relevant.
    Padded target positions with item ID 0 are ignored.
    """

    # The first six candidate positions hold basket targets.
    # True = a real positive item; False = padding or negative.
    relevance = torch.zeros_like(
        candidate_item_ids,
        dtype=torch.float32,
    )

    relevance[:, :MAX_ITEMS_PER_BASKET] = (
        target_items.ne(0).float()
    )

    ranking_order = scores.argsort(
        dim=1,
        descending=True,
    )

    ranked_relevance = relevance.gather(
        dim=1,
        index=ranking_order,
    )

    positive_count = relevance.sum(dim=1)

    for k in RANKING_K_VALUES:
        top_k_relevance = ranked_relevance[:, :k]

        # Recall: what fraction of all correct basket items
        # appeared in the top K recommendations?
        recall_per_example = (
            top_k_relevance.sum(dim=1) / positive_count
        )

        # NDCG: correct recommendations near rank 1 get more credit.
        discounts = 1.0 / torch.log2(
            torch.arange(
                2,
                k + 2,
                device=scores.device,
                dtype=torch.float32,
            )
        )

        dcg = (
            top_k_relevance * discounts.unsqueeze(0)
        ).sum(dim=1)

        ideal_dcg_lookup = torch.cat(
            [
                torch.zeros(
                    1,
                    device=scores.device,
                ),
                discounts.cumsum(dim=0),
            ]
        )

        ideal_dcg = ideal_dcg_lookup[
            positive_count.long()
        ]

        ndcg_per_example = dcg / ideal_dcg

        metric_totals[f"recall@{k}"] += (
            recall_per_example.sum().item()
        )

        metric_totals[f"ndcg@{k}"] += (
            ndcg_per_example.sum().item()
        )


# ------------------------------------------------------------
# 3. Build popularity baseline from training baskets only
# ------------------------------------------------------------
train_basket_mask = model_baskets["split"].eq(
    "train"
).to_numpy()

train_item_ids = basket_item_matrix[
    train_basket_mask
].reshape(-1)

item_popularity_counts = np.bincount(
    train_item_ids,
    minlength=NUM_TRAIN_ITEMS + 1,
).astype(np.float32)

# Item 0 is padding and must never receive a recommendation score.
item_popularity_counts[0] = -np.inf

item_popularity_scores = torch.from_numpy(
    item_popularity_counts
).to(device)

print(
    "Popularity baseline built from",
    int(train_basket_mask.sum()),
    "training baskets.",
)


# ------------------------------------------------------------
# 4. Evaluate either popularity or SASRec
# ------------------------------------------------------------
@torch.inference_mode()
def evaluate_validation_ranking(model_name):
    metric_totals = {
        "recall@10": 0.0,
        "ndcg@10": 0.0,
        "recall@20": 0.0,
        "ndcg@20": 0.0,
    }

    example_count = 0
    gpu_number = torch.cuda.current_device()

    # This seed is reset for each model.
    # Therefore, popularity and SASRec see identical candidates.
    with torch.random.fork_rng(devices=[gpu_number]):
        torch.manual_seed(EVALUATION_SEED)
        torch.cuda.manual_seed_all(EVALUATION_SEED)

        for batch in validation_loader:
            history_items = batch["history_items"].to(
                device,
                non_blocking=True,
            )

            target_items = batch["target_items"].to(
                device,
                non_blocking=True,
            )

            assert target_items.ne(0).any(dim=1).all()

            negative_items = sample_distinct_negative_items(
                target_items=target_items,
                num_items=NUM_TRAIN_ITEMS,
                num_negative_items=NUM_EVALUATION_NEGATIVES,
            )

            # First positions = true items; remaining = negatives.
            candidate_item_ids = torch.cat(
                [
                    target_items,
                    negative_items,
                ],
                dim=1,
            )

            if model_name == "popularity":
                scores = item_popularity_scores[
                    candidate_item_ids
                ]

            elif model_name == "sasrec":
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.bfloat16,
                ):
                    latest_context, _, _ = sasrec_model(
                        history_items
                    )

                    scores = item_scorer(
                        latest_context,
                        candidate_item_ids,
                    )

                scores = scores.float()

            else:
                raise ValueError(
                    "model_name must be 'popularity' or 'sasrec'."
                )

            # Target padding slots must never rank as recommendations.
            scores = scores.masked_fill(
                candidate_item_ids.eq(0),
                float("-inf"),
            )

            update_ranking_totals(
                scores=scores,
                candidate_item_ids=candidate_item_ids,
                target_items=target_items,
                metric_totals=metric_totals,
            )

            example_count += target_items.size(0)

    return {
        "model": model_name,
        "examples": example_count,
        **{
            metric_name: metric_total / example_count
            for metric_name, metric_total in metric_totals.items()
        },
    }


# ------------------------------------------------------------
# 5. Run validation ranking comparison
# ------------------------------------------------------------
popularity_validation_results = evaluate_validation_ranking(
    "popularity"
)

sasrec_validation_results = evaluate_validation_ranking(
    "sasrec"
)

validation_ranking_results = pd.DataFrame(
    [
        popularity_validation_results,
        sasrec_validation_results,
    ]
)

VALIDATION_METRICS_PATH = (
    MODEL_DIR / "validation_ranking_metrics.csv"
)

validation_ranking_results.to_csv(
    VALIDATION_METRICS_PATH,
    index=False,
)

candidate_protocol = pd.DataFrame(
    [
        {
            "evaluation_split": "validation",
            "catalogue": "training-only known items",
            "catalogue_item_count": NUM_TRAIN_ITEMS,
            "positive_rule": (
                "all known items in the next timestamp basket"
            ),
            "negative_rule": (
                "100 distinct sampled training-catalogue items; "
                "never a true target item"
            ),
            "ranking_metrics": "Recall@10, NDCG@10, Recall@20, NDCG@20",
            "evaluation_seed": EVALUATION_SEED,
            "test_set_used": False,
        }
    ]
)

CANDIDATE_PROTOCOL_PATH = (
    MODEL_DIR / "validation_candidate_protocol.csv"
)

candidate_protocol.to_csv(
    CANDIDATE_PROTOCOL_PATH,
    index=False,
)

display(validation_ranking_results)

print("✅ Validation metrics saved:", VALIDATION_METRICS_PATH)
print("✅ Candidate protocol saved:", CANDIDATE_PROTOCOL_PATH)
print("✅ Test set remains untouched.")

Loaded best checkpoint from epoch: 5
Best validation sampled loss: 2.5383
Popularity baseline built from 1927110 training baskets.


,model,examples,recall@10,ndcg@10,recall@20,ndcg@20
0,popularity,182936,0.616703,0.374639,0.775275,0.414825
1,sasrec,182936,0.765012,0.667342,0.835737,0.685232


✅ Validation metrics saved: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\validation_ranking_metrics.csv
✅ Candidate protocol saved: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\validation_candidate_protocol.csv
✅ Test set remains untouched.


In [20]:
import time
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Required objects from earlier cells:
# sasrec_model, train_dataset, validation_dataset,
# multi_positive_loss, evaluate_validation_loss,
# evaluate_validation_ranking, device
# ------------------------------------------------------------
assert "sasrec_model" in globals()
assert "train_dataset" in globals()
assert "validation_dataset" in globals()
assert "multi_positive_loss" in globals()
assert "evaluate_validation_loss" in globals()
assert "evaluate_validation_ranking" in globals()

PROJECT_ROOT = Path(
    r"E:\ALL MCP Servers\Recommendation"
)

MODEL_DIR = PROJECT_ROOT / "models" / "sasrec_basket_v1"

BEST_MODEL_PATH = MODEL_DIR / "best_model.pt"
HISTORY_PATH = MODEL_DIR / "training_history.csv"
VALIDATION_METRICS_PATH = (
    MODEL_DIR / "validation_ranking_metrics.csv"
)

CONTINUED_MODEL_PATH = (
    MODEL_DIR / "best_model_by_ndcg10.pt"
)

CONTINUED_HISTORY_PATH = (
    MODEL_DIR / "continued_training_history.csv"
)

# ------------------------------------------------------------
# 1. Load the best previous model: Epoch 5
# ------------------------------------------------------------
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    checkpoint["model_state_dict"]
)

START_EPOCH = int(checkpoint["epoch"]) + 1
FINAL_EPOCH = 15

print("Loaded checkpoint from Epoch:", checkpoint["epoch"])
print("Continuing from Epoch:", START_EPOCH)

# ------------------------------------------------------------
# 2. Fine-tuning configuration
# ------------------------------------------------------------
# Smaller learning rate = gentler refinement of learned weights.
FINE_TUNE_LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
TRAIN_BATCH_SIZE = 256
VALIDATION_BATCH_SIZE = 512
EARLY_STOPPING_PATIENCE = 2
GRADIENT_CLIP_NORM = 1.0
MIN_NDCG_IMPROVEMENT = 1e-4

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

# New optimizer starts a fine-tuning phase from saved model weights.
optimizer = torch.optim.AdamW(
    sasrec_model.parameters(),
    lr=FINE_TUNE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Existing Epoch 1–5 history remains preserved.
previous_history = pd.read_csv(HISTORY_PATH)

# SASRec validation ranking from the existing Epoch 5 evaluation.
previous_ranking = pd.read_csv(
    VALIDATION_METRICS_PATH
)

epoch_5_ndcg10 = float(
    previous_ranking.loc[
        previous_ranking["model"].eq("sasrec"),
        "ndcg@10",
    ].iloc[0]
)

best_ndcg10 = epoch_5_ndcg10
epochs_without_improvement = 0
continued_history = []

print(f"Starting NDCG@10 from Epoch 5: {best_ndcg10:.6f}")


# ------------------------------------------------------------
# 3. Reusable functions
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, loss_function):
    model.train()

    weighted_loss_sum = 0.0
    positive_pair_count = 0
    epoch_start = time.perf_counter()

    for step, batch in enumerate(loader, start=1):
        history_items = batch["history_items"].to(
            device,
            non_blocking=True,
        )

        target_items = batch["target_items"].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            latest_context, _, _ = model(history_items)

            loss, loss_info = loss_function(
                latest_context,
                target_items,
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP_NORM,
        )

        optimizer.step()

        pair_count = loss_info["positive_target_pairs"]

        weighted_loss_sum += loss.item() * pair_count
        positive_pair_count += pair_count

        if step % 500 == 0:
            print(
                f"  Step {step:,}/{len(loader):,} | "
                f"training loss: "
                f"{weighted_loss_sum / positive_pair_count:.4f}"
            )

    return (
        weighted_loss_sum / positive_pair_count,
        time.perf_counter() - epoch_start,
    )


def save_ndcg_checkpoint(epoch, ndcg10, validation_loss):
    torch.save(
        {
            "optimizer_state_dict": optimizer.state_dict(),
            "model_state_dict": sasrec_model.state_dict(),
            "epoch": epoch,
            "validation_ndcg@10": ndcg10,
            "validation_sampled_loss": validation_loss,
            "config": {
                "phase": "continued fine-tuning",
                "learning_rate": FINE_TUNE_LEARNING_RATE,
                "training_batch_size": TRAIN_BATCH_SIZE,
                "selection_metric": "validation NDCG@10",
            },
        },
        CONTINUED_MODEL_PATH,
    )


# ------------------------------------------------------------
# 4. Continue training: Epochs 6–10
# ------------------------------------------------------------
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for epoch in range(START_EPOCH, FINAL_EPOCH + 1):
    print(f"\n{'=' * 60}")
    print(f"Fine-tuning Epoch {epoch}/{FINAL_EPOCH}")

    train_loss, epoch_seconds = train_one_epoch(
        sasrec_model,
        train_loader,
        optimizer,
        multi_positive_loss,
    )

    validation_loss = evaluate_validation_loss(
        sasrec_model,
        validation_loader,
        multi_positive_loss,
    )

    ranking_result = evaluate_validation_ranking(
        "sasrec"
    )

    current_ndcg10 = ranking_result["ndcg@10"]
    current_recall10 = ranking_result["recall@10"]

    improved = (
        current_ndcg10
        > best_ndcg10 + MIN_NDCG_IMPROVEMENT
    )

    record = {
        "epoch": epoch,
        "train_sampled_loss": train_loss,
        "validation_sampled_loss": validation_loss,
        "validation_recall@10": current_recall10,
        "validation_ndcg@10": current_ndcg10,
        "epoch_seconds": epoch_seconds,
        "gpu_peak_allocated_gb": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        "improved_ndcg@10": improved,
    }

    continued_history.append(record)

    print(f"Train loss:      {train_loss:.4f}")
    print(f"Validation loss: {validation_loss:.4f}")
    print(f"Validation Recall@10: {current_recall10:.6f}")
    print(f"Validation NDCG@10:   {current_ndcg10:.6f}")

    if improved:
        best_ndcg10 = current_ndcg10
        epochs_without_improvement = 0

        save_ndcg_checkpoint(
            epoch=epoch,
            ndcg10=current_ndcg10,
            validation_loss=validation_loss,
        )

        print("✅ New best NDCG@10 checkpoint saved.")

    else:
        epochs_without_improvement += 1

        print(
            "No NDCG@10 improvement: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print("⏹️ Early stopping activated.")
            break


# ------------------------------------------------------------
# 5. Save combined readable history
# ------------------------------------------------------------
continued_history_df = pd.DataFrame(
    continued_history
)

combined_history = pd.concat(
    [
        previous_history,
        continued_history_df,
    ],
    ignore_index=True,
    sort=False,
)

combined_history.to_csv(
    CONTINUED_HISTORY_PATH,
    index=False,
)

display(continued_history_df)

print("\n✅ Continued training finished.")
print("Best validation NDCG@10:", round(best_ndcg10, 6))
print("Best model by NDCG@10:", CONTINUED_MODEL_PATH)
print("Combined history:", CONTINUED_HISTORY_PATH)
print("✅ Test set remains untouched.")

Loaded checkpoint from Epoch: 1
Continuing from Epoch: 2
Starting NDCG@10 from Epoch 5: 0.667342

Fine-tuning Epoch 2/15
  Step 500/3,704 | training loss: 1.6590
  Step 1,000/3,704 | training loss: 1.6558
  Step 1,500/3,704 | training loss: 1.6514
  Step 2,000/3,704 | training loss: 1.6483
  Step 2,500/3,704 | training loss: 1.6490
  Step 3,000/3,704 | training loss: 1.6484
  Step 3,500/3,704 | training loss: 1.6452
Train loss:      1.6450
Validation loss: 2.5150
Validation Recall@10: 0.769673
Validation NDCG@10:   0.674873
✅ New best NDCG@10 checkpoint saved.

Fine-tuning Epoch 3/15
  Step 500/3,704 | training loss: 1.5479
  Step 1,000/3,704 | training loss: 1.5467
  Step 1,500/3,704 | training loss: 1.5480
  Step 2,000/3,704 | training loss: 1.5484
  Step 2,500/3,704 | training loss: 1.5462
  Step 3,000/3,704 | training loss: 1.5450
  Step 3,500/3,704 | training loss: 1.5435
Train loss:      1.5427
Validation loss: 2.5017
Validation Recall@10: 0.772080
Validation NDCG@10:   0.680471


,epoch,train_sampled_loss,validation_sampled_loss,validation_recall@10,validation_ndcg@10,epoch_seconds,gpu_peak_allocated_gb,improved_ndcg@10
0,2,1.645047,2.514976,0.769673,0.674873,55.399314,0.577896,True
1,3,1.542713,2.501708,0.772080,0.680471,55.530090,0.577896,True
2,4,1.447458,2.486280,0.776493,0.686979,55.586785,0.577896,True
3,5,1.356893,2.477822,0.778115,0.691501,55.677987,0.577896,True
4,6,1.271431,2.471349,0.781102,0.696973,56.959744,0.577896,True
5,7,1.190442,2.472993,0.783259,0.701084,54.923407,0.577896,True
6,8,1.113769,2.470913,0.786017,0.705453,56.330180,0.577896,True
7,9,1.041723,2.475231,0.786621,0.708171,56.726738,0.577896,True
8,10,0.975132,2.476896,0.789762,0.711889,55.636508,0.577896,True
9,11,0.912022,2.488555,0.790024,0.713935,57.845712,0.577896,True



✅ Continued training finished.
Best validation NDCG@10: 0.72436
Best model by NDCG@10: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\best_model_by_ndcg10.pt
Combined history: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\continued_training_history.csv
✅ Test set remains untouched.


In [21]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 1. Load the validation-selected Epoch 15 model
# ------------------------------------------------------------
FINAL_SASREC_PATH = (
    MODEL_DIR / "best_model_by_ndcg10.pt"
)

final_checkpoint = torch.load(
    FINAL_SASREC_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    final_checkpoint["model_state_dict"]
)

sasrec_model.eval()

print("Loaded final SASRec checkpoint.")
print("Selected epoch:", final_checkpoint["epoch"])
print(
    "Validation NDCG@10:",
    round(final_checkpoint["validation_ndcg@10"], 6),
)

# Test is never shuffled.
test_evaluation_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

# ------------------------------------------------------------
# 2. Same candidate protocol as validation
# ------------------------------------------------------------
NUM_TEST_NEGATIVES = 100
TEST_EVALUATION_SEED = 30_042
RANKING_K_VALUES = [10, 20]


def sample_distinct_negative_items(
    target_items,
    num_items,
    num_negative_items,
):
    batch_size = target_items.size(0)

    negative_items = torch.randint(
        low=1,
        high=num_items + 1,
        size=(batch_size, num_negative_items),
        device=target_items.device,
    )

    while True:
        target_collision = (
            negative_items.unsqueeze(-1)
            .eq(target_items.unsqueeze(1))
            .any(dim=-1)
        )

        duplicate_collision = torch.zeros_like(
            negative_items,
            dtype=torch.bool,
        )

        for position in range(1, num_negative_items):
            duplicate_collision[:, position] = (
                negative_items[:, position].unsqueeze(1)
                .eq(negative_items[:, :position])
                .any(dim=1)
            )

        invalid_negative = (
            target_collision | duplicate_collision
        )

        if not invalid_negative.any():
            break

        negative_items[invalid_negative] = torch.randint(
            low=1,
            high=num_items + 1,
            size=(int(invalid_negative.sum().item()),),
            device=target_items.device,
        )

    return negative_items


def update_ranking_totals(
    scores,
    candidate_item_ids,
    target_items,
    metric_totals,
):
    relevance = torch.zeros_like(
        candidate_item_ids,
        dtype=torch.float32,
    )

    relevance[:, :MAX_ITEMS_PER_BASKET] = (
        target_items.ne(0).float()
    )

    ranking_order = scores.argsort(
        dim=1,
        descending=True,
    )

    ranked_relevance = relevance.gather(
        dim=1,
        index=ranking_order,
    )

    positive_count = relevance.sum(dim=1)

    for k in RANKING_K_VALUES:
        top_k_relevance = ranked_relevance[:, :k]

        recall_per_example = (
            top_k_relevance.sum(dim=1) / positive_count
        )

        discounts = 1.0 / torch.log2(
            torch.arange(
                2,
                k + 2,
                device=scores.device,
                dtype=torch.float32,
            )
        )

        dcg = (
            top_k_relevance * discounts.unsqueeze(0)
        ).sum(dim=1)

        ideal_dcg_lookup = torch.cat(
            [
                torch.zeros(1, device=scores.device),
                discounts.cumsum(dim=0),
            ]
        )

        ideal_dcg = ideal_dcg_lookup[
            positive_count.long()
        ]

        ndcg_per_example = dcg / ideal_dcg

        metric_totals[f"recall@{k}"] += (
            recall_per_example.sum().item()
        )

        metric_totals[f"ndcg@{k}"] += (
            ndcg_per_example.sum().item()
        )


# ------------------------------------------------------------
# 3. Popularity scores from training data only
# ------------------------------------------------------------
train_basket_mask = model_baskets["split"].eq(
    "train"
).to_numpy()

train_item_ids = basket_item_matrix[
    train_basket_mask
].reshape(-1)

popularity_counts = np.bincount(
    train_item_ids,
    minlength=NUM_TRAIN_ITEMS + 1,
).astype(np.float32)

popularity_counts[0] = -np.inf

popularity_scores = torch.from_numpy(
    popularity_counts
).to(device)


# ------------------------------------------------------------
# 4. Test evaluator: popularity or final SASRec
# ------------------------------------------------------------
@torch.inference_mode()
def evaluate_test_ranking(model_name):
    metric_totals = {
        "recall@10": 0.0,
        "ndcg@10": 0.0,
        "recall@20": 0.0,
        "ndcg@20": 0.0,
    }

    example_count = 0
    gpu_number = torch.cuda.current_device()

    # Same candidates for popularity and SASRec.
    with torch.random.fork_rng(devices=[gpu_number]):
        torch.manual_seed(TEST_EVALUATION_SEED)
        torch.cuda.manual_seed_all(TEST_EVALUATION_SEED)

        for batch in test_evaluation_loader:
            history_items = batch["history_items"].to(
                device,
                non_blocking=True,
            )

            target_items = batch["target_items"].to(
                device,
                non_blocking=True,
            )

            negative_items = sample_distinct_negative_items(
                target_items=target_items,
                num_items=NUM_TRAIN_ITEMS,
                num_negative_items=NUM_TEST_NEGATIVES,
            )

            candidate_item_ids = torch.cat(
                [
                    target_items,
                    negative_items,
                ],
                dim=1,
            )

            if model_name == "popularity":
                scores = popularity_scores[
                    candidate_item_ids
                ]

            elif model_name == "sasrec":
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.bfloat16,
                ):
                    latest_context, _, _ = sasrec_model(
                        history_items
                    )

                    scores = item_scorer(
                        latest_context,
                        candidate_item_ids,
                    )

                scores = scores.float()

            else:
                raise ValueError(
                    "Use 'popularity' or 'sasrec'."
                )

            # Target basket padding must never rank.
            scores = scores.masked_fill(
                candidate_item_ids.eq(0),
                float("-inf"),
            )

            update_ranking_totals(
                scores=scores,
                candidate_item_ids=candidate_item_ids,
                target_items=target_items,
                metric_totals=metric_totals,
            )

            example_count += target_items.size(0)

    return {
        "model": model_name,
        "test_examples": example_count,
        **{
            metric_name: total / example_count
            for metric_name, total in metric_totals.items()
        },
    }


# ------------------------------------------------------------
# 5. Run final test evaluation once
# ------------------------------------------------------------
test_popularity_result = evaluate_test_ranking(
    "popularity"
)

test_sasrec_result = evaluate_test_ranking(
    "sasrec"
)

final_test_results = pd.DataFrame(
    [
        test_popularity_result,
        test_sasrec_result,
    ]
)

FINAL_TEST_METRICS_PATH = (
    MODEL_DIR / "final_sasrec_test_metrics.csv"
)

final_test_results.to_csv(
    FINAL_TEST_METRICS_PATH,
    index=False,
)

test_protocol = pd.DataFrame(
    [
        {
            "evaluation_split": "test",
            "test_examples": len(test_dataset),
            "catalogue": "training-only known items",
            "catalogue_item_count": NUM_TRAIN_ITEMS,
            "positive_rule": (
                "all known items in the next timestamp basket"
            ),
            "negative_rule": (
                "100 distinct sampled training-catalogue items; "
                "never a target item"
            ),
            "metrics": "Recall@10, NDCG@10, Recall@20, NDCG@20",
            "evaluation_seed": TEST_EVALUATION_SEED,
            "selected_model_epoch": int(
                final_checkpoint["epoch"]
            ),
            "test_item_known_coverage_percent": 89.18,
        }
    ]
)

FINAL_TEST_PROTOCOL_PATH = (
    MODEL_DIR / "final_sasrec_test_protocol.csv"
)

test_protocol.to_csv(
    FINAL_TEST_PROTOCOL_PATH,
    index=False,
)

display(final_test_results)

print("✅ Final SASRec test metrics:", FINAL_TEST_METRICS_PATH)
print("✅ Final test protocol:", FINAL_TEST_PROTOCOL_PATH)

Loaded final SASRec checkpoint.
Selected epoch: 15
Validation NDCG@10: 0.72436


,model,test_examples,recall@10,ndcg@10,recall@20,ndcg@20
0,popularity,173786,0.585501,0.352292,0.750052,0.393941
1,sasrec,173786,0.773758,0.698580,0.828870,0.712470


✅ Final SASRec test metrics: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\final_sasrec_test_metrics.csv
✅ Final test protocol: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\final_sasrec_test_protocol.csv


In [22]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(
    r"E:\ALL MCP Servers\Recommendation"
)

MODEL_DIR = PROJECT_ROOT / "models" / "sasrec_basket_v1"

FINAL_SASREC_PATH = (
    MODEL_DIR / "best_model_by_ndcg10.pt"
)

CONTEXT_VECTOR_FILE = (
    MODEL_DIR / "final_sasrec_test_context_vectors.npy"
)

CONTEXT_METADATA_FILE = (
    MODEL_DIR / "final_sasrec_test_context_metadata.csv"
)

CONTEXT_MANIFEST_FILE = (
    MODEL_DIR / "final_sasrec_test_context_manifest.csv"
)

# Load the fixed final Epoch 15 model.
checkpoint = torch.load(
    FINAL_SASREC_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    checkpoint["model_state_dict"]
)

sasrec_model.eval()

PREDICTION_BATCH_SIZE = 512

context_loader = DataLoader(
    test_dataset,
    batch_size=PREDICTION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

assert len(test_dataset) == len(test_examples)

# Preallocate on disk:
# [173,786 test examples, 64 embedding values]
context_store = np.lib.format.open_memmap(
    CONTEXT_VECTOR_FILE,
    mode="w+",
    dtype=np.float32,
    shape=(
        len(test_dataset),
        EMBEDDING_DIM,
    ),
)

start_time = time.perf_counter()
next_row = 0

with torch.inference_mode():
    for batch_number, batch in enumerate(
        context_loader,
        start=1,
    ):
        history_items = batch["history_items"].to(
            device,
            non_blocking=True,
        )

        # BF16 makes prediction fast and memory-efficient.
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            context_vectors, _, _ = sasrec_model(
                history_items
            )

        batch_size = context_vectors.size(0)

        # Save compact context vectors as float32.
        context_store[
            next_row:next_row + batch_size
        ] = context_vectors.float().cpu().numpy()

        next_row += batch_size

        if batch_number % 50 == 0:
            print(
                f"Processed {next_row:,}/"
                f"{len(test_dataset):,} test examples"
            )

context_store.flush()

assert next_row == len(test_dataset)

# Save the row identity and true next-basket reference.
context_metadata = test_examples[
    [
        "visitorid",
        "target_timestamp",
        "target_block_id",
        "history_length",
        "target_item_indices",
        "target_item_count",
    ]
].copy()

context_metadata.insert(
    0,
    "context_row_index",
    np.arange(
        len(context_metadata),
        dtype=np.int64,
    ),
)

context_metadata.to_csv(
    CONTEXT_METADATA_FILE,
    index=False,
)

elapsed_seconds = time.perf_counter() - start_time

context_manifest = pd.DataFrame(
    [
        {
            "checkpoint_epoch": int(checkpoint["epoch"]),
            "test_examples": len(test_dataset),
            "context_vector_shape": (
                f"({len(test_dataset)}, {EMBEDDING_DIM})"
            ),
            "vector_dtype": "float32",
            "context_file_size_mb": round(
                CONTEXT_VECTOR_FILE.stat().st_size
                / 1024**2,
                2,
            ),
            "elapsed_seconds": round(
                elapsed_seconds,
                2,
            ),
            "context_file": str(CONTEXT_VECTOR_FILE),
            "metadata_file": str(CONTEXT_METADATA_FILE),
        }
    ]
)

context_manifest.to_csv(
    CONTEXT_MANIFEST_FILE,
    index=False,
)

print("\n✅ Saved compact contexts for all test examples.")
print("Context file:", CONTEXT_VECTOR_FILE)
print("Metadata file:", CONTEXT_METADATA_FILE)

display(context_manifest)
display(context_metadata.head())

Processed 25,600/173,786 test examples
Processed 51,200/173,786 test examples
Processed 76,800/173,786 test examples
Processed 102,400/173,786 test examples
Processed 128,000/173,786 test examples
Processed 153,600/173,786 test examples

✅ Saved compact contexts for all test examples.
Context file: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\final_sasrec_test_context_vectors.npy
Metadata file: E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\final_sasrec_test_context_metadata.csv


,checkpoint_epoch,test_examples,context_vector_shape,vector_dtype,context_file_size_mb,elapsed_seconds,context_file,metadata_file
0,15,173786,"(173786, 64)",float32,42.43,2.77,E:\ALL MCP Servers\Recommendation\models\sasre...,E:\ALL MCP Servers\Recommendation\models\sasre...


,context_row_index,visitorid,target_timestamp,target_block_id,history_length,target_item_indices,target_item_count
0,0,0,1442004759591,v0_t1442004759591,1,[154167],1
1,1,0,1442004917175,v0_t1442004917175,2,[28938],1
2,2,6,1440916778318,v6_t1440916778318,1,[109356],1
3,3,6,1440916823805,v6_t1440916823805,2,[148518],1
4,4,6,1440917649385,v6_t1440917649385,3,[148518],1


In [23]:
import numpy as np
import pandas as pd
import torch

# Load saved compact test contexts without loading all into RAM.
saved_contexts = np.load(
    CONTEXT_VECTOR_FILE,
    mmap_mode="r",
)

saved_metadata = pd.read_csv(
    CONTEXT_METADATA_FILE
)

assert saved_contexts.shape == (
    len(saved_metadata),
    EMBEDDING_DIM,
)

# Load original Retailrocket item IDs for readable output.
REFRAME_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
    / "training_reframe"
)

item_mapping = pd.read_csv(
    REFRAME_DIR / "train_item_id_mapping.csv"
)

item_index_values = item_mapping[
    "item_index"
].to_numpy()

original_itemid_values = item_mapping[
    "itemid"
].to_numpy()

# Item matrix already exists inside the final SASRec model.
catalogue_item_vectors = (
    sasrec_model
    .basket_encoder
    .item_embedding
    .weight[1:]
)

SCORE_SCALE = EMBEDDING_DIM ** -0.5


@torch.inference_mode()
def all_catalogue_scores(context_row_index):
    """
    Reconstructs one score for every one of the
    200,978 training-known items.
    """

    context_vector = torch.from_numpy(
        np.asarray(
            saved_contexts[
                context_row_index:
                context_row_index + 1
            ]
        )
    ).to(
        device,
        dtype=torch.float32,
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        logits = torch.matmul(
            context_vector,
            catalogue_item_vectors.transpose(0, 1),
        ) * SCORE_SCALE

    return logits.squeeze(0).float().cpu().numpy()


# Choose any test example row you want to inspect.
CONTEXT_ROW_INDEX = 0

all_scores = all_catalogue_scores(
    CONTEXT_ROW_INDEX
)

# One row per catalogue item: this is the full prediction vector
# for this one test history.
full_prediction_for_one_example = pd.DataFrame(
    {
        "item_index": item_index_values,
        "itemid": original_itemid_values,
        "logit_score": all_scores,
    }
)

example_information = saved_metadata.iloc[
    CONTEXT_ROW_INDEX
]

print("Test context row:", CONTEXT_ROW_INDEX)
print("Visitor:", example_information["visitorid"])
print(
    "Target block:",
    example_information["target_block_id"],
)

print("\nFull score-vector shape:")
print(full_prediction_for_one_example.shape)

display(full_prediction_for_one_example.head(20))

Test context row: 0
Visitor: 0
Target block: v0_t1442004759591

Full score-vector shape:
(200978, 3)


C:\Users\ayyub\AppData\Local\Temp\ipykernel_32592\643554025.py:59: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  context_vector = torch.from_numpy(


,item_index,itemid,logit_score
0,1,4,-8.937500
1,2,6,-3.546875
2,3,9,-7.062500
3,4,15,5.031250
4,5,16,-3.656250
5,6,17,-1.687500
6,7,19,-1.671875
7,8,22,0.855469
8,9,24,-5.968750
9,10,25,-1.828125


In [25]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    r"E:\ALL MCP Servers\Recommendation"
)

REFRAME_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
    / "training_reframe"
)

# Rebuild internal model-index → original Retailrocket item-ID lookup.
item_mapping = pd.read_csv(
    REFRAME_DIR / "train_item_id_mapping.csv"
)

index_to_itemid = np.zeros(
    NUM_TRAIN_ITEMS + 1,
    dtype=np.int64,
)

index_to_itemid[
    item_mapping["item_index"].to_numpy()
] = item_mapping["itemid"].to_numpy()

# Actual next item(s).
actual_item_indices = json.loads(
    example_information["target_item_indices"]
)

actual_itemids = [
    int(index_to_itemid[item_index])
    for item_index in actual_item_indices
]

# Rank-1 model prediction = highest logit score.
best_prediction = full_prediction_for_one_example.loc[
    full_prediction_for_one_example["logit_score"].idxmax()
]

predicted_itemid = int(best_prediction["itemid"])
predicted_score = float(
    best_prediction["logit_score"]
)

clean_prediction_output = pd.DataFrame(
    [
        {
            "visitorid": int(
                example_information["visitorid"]
            ),
            "target_timestamp": int(
                example_information["target_timestamp"]
            ),
            "expected_next_itemids": actual_itemids,
            "predicted_next_itemid": predicted_itemid,
            "predicted_logit_score": round(
                predicted_score,
                6,
            ),
            "prediction_correct": (
                predicted_itemid in actual_itemids
            ),
        }
    ]
)

display(clean_prediction_output)

,visitorid,target_timestamp,expected_next_itemids,predicted_next_itemid,predicted_logit_score,prediction_correct
0,0,1442004759591,[357564],285930,20.125,False


In [29]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ============================================================
# CONTROLLERS — change these whenever you want
# ============================================================
CONTEXT_ROW_INDEX = 0
NUMBER_OF_RESULTS = 50
SHOW_ONLY_TRUE_PREDICTIONS = True
# ============================================================

PROJECT_ROOT = Path(r"E:\ALL MCP Servers\Recommendation")

MODEL_DIR = PROJECT_ROOT / "models" / "sasrec_basket_v1"

REFRAME_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
    / "training_reframe"
)

CONTEXT_VECTOR_FILE = (
    MODEL_DIR / "final_sasrec_test_context_vectors.npy"
)

CONTEXT_METADATA_FILE = (
    MODEL_DIR / "final_sasrec_test_context_metadata.csv"
)

FINAL_SASREC_PATH = MODEL_DIR / "best_model_by_ndcg10.pt"

# CSV output directory.
OUTPUT_DIR = MODEL_DIR / "test_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Your SASRec model class must already have been created earlier.
assert "sasrec_model" in globals(), (
    "Run the earlier SASRec model-definition cell first."
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# LOAD THE TRAINED MODEL
# ============================================================
checkpoint = torch.load(
    FINAL_SASREC_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    checkpoint["model_state_dict"]
)

sasrec_model.to(device)
sasrec_model.eval()

# ============================================================
# LOAD TEST CONTEXTS
# ============================================================
context_metadata = pd.read_csv(
    CONTEXT_METADATA_FILE
)

saved_contexts = np.load(
    CONTEXT_VECTOR_FILE,
    mmap_mode="r",
)

assert len(context_metadata) == len(saved_contexts)
assert 0 <= CONTEXT_ROW_INDEX < len(context_metadata)

# ============================================================
# LOAD ITEM-ID MAPPING
# ============================================================
item_mapping = pd.read_csv(
    REFRAME_DIR / "train_item_id_mapping.csv"
)

NUM_TRAIN_ITEMS = int(
    item_mapping["item_index"].max()
)

index_to_itemid = np.zeros(
    NUM_TRAIN_ITEMS + 1,
    dtype=np.int64,
)

index_to_itemid[
    item_mapping["item_index"].to_numpy(dtype=np.int64)
] = item_mapping["itemid"].to_numpy(dtype=np.int64)

assert (index_to_itemid[1:] != 0).all()

# ============================================================
# SELECT TEST EXAMPLE
# ============================================================
example_information = context_metadata.iloc[
    CONTEXT_ROW_INDEX
]

context_vector = torch.from_numpy(
    np.asarray(
        saved_contexts[
            CONTEXT_ROW_INDEX:CONTEXT_ROW_INDEX + 1
        ]
    ).copy()
).to(
    device,
    dtype=torch.float32,
)

# ============================================================
# SCORE ALL TRAINING-KNOWN ITEMS
# ============================================================
with torch.inference_mode():
    catalogue_item_vectors = (
        sasrec_model.basket_encoder.item_embedding.weight[1:]
    )

    full_scores = torch.matmul(
        context_vector,
        catalogue_item_vectors.T,
    ) * (catalogue_item_vectors.size(1) ** -0.5)

full_scores = full_scores.squeeze(0).float().cpu()

NUMBER_OF_RESULTS = min(
    NUMBER_OF_RESULTS,
    NUM_TRAIN_ITEMS,
)

# Retrieve the requested highest-scoring predictions.
top_scores, top_positions = torch.topk(
    full_scores,
    k=NUMBER_OF_RESULTS,
)

# Add 1 because index 0 is reserved for padding.
top_item_indices = top_positions.numpy() + 1

# ============================================================
# GET ACTUAL NEXT ITEM(S)
# ============================================================
actual_item_indices = json.loads(
    example_information["target_item_indices"]
)

actual_itemids = [
    int(index_to_itemid[item_index])
    for item_index in actual_item_indices
]

# ============================================================
# SUMMARY OUTPUT
# ============================================================
predicted_itemid = int(
    index_to_itemid[top_item_indices[0]]
)

predicted_score = float(top_scores[0])

clean_prediction_output = pd.DataFrame(
    [
        {
            "context_row_index": CONTEXT_ROW_INDEX,
            "visitorid": int(
                example_information["visitorid"]
            ),
            "target_timestamp": int(
                example_information["target_timestamp"]
            ),
            "expected_next_itemids": json.dumps(
                actual_itemids
            ),
            "predicted_next_itemid": predicted_itemid,
            "predicted_logit_score": round(
                predicted_score,
                6,
            ),
            "top_1_prediction_correct": (
                predicted_itemid in actual_itemids
            ),
        }
    ]
)

# ============================================================
# RANKED PREDICTIONS OUTPUT
# ============================================================
ranked_prediction_output = pd.DataFrame(
    {
        "rank": np.arange(
            1,
            NUMBER_OF_RESULTS + 1,
        ),
        "predicted_item_index": top_item_indices,
        "predicted_itemid": index_to_itemid[
            top_item_indices
        ],
        "logit_score": top_scores.numpy().round(6),
    }
)

ranked_prediction_output["is_actual_next_item"] = (
    ranked_prediction_output["predicted_itemid"]
    .isin(actual_itemids)
)

# ============================================================
# FILTER: SHOW ONLY ACTUAL NEXT-BASKET ITEMS
# ============================================================
if SHOW_ONLY_TRUE_PREDICTIONS:
    ranked_prediction_output = (
        ranked_prediction_output
        .loc[
            ranked_prediction_output[
                "is_actual_next_item"
            ]
        ]
        .copy()
    )

    print(
        "Showing only correct recommendations "
        "found inside the selected Top-N results."
    )

    if ranked_prediction_output.empty:
        print(
            "No actual next item appeared in the selected "
            f"Top-{NUMBER_OF_RESULTS} recommendations."
        )

# ============================================================
# SAVE CSV FILES WITH TIMESTAMP
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary_csv_path = (
    OUTPUT_DIR / f"test_summary_{timestamp}.csv"
)

ranked_csv_path = (
    OUTPUT_DIR / f"test_ranked_predictions_{timestamp}.csv"
)

clean_prediction_output.to_csv(
    summary_csv_path,
    index=False,
)

ranked_prediction_output.to_csv(
    ranked_csv_path,
    index=False,
)

# ============================================================
# DISPLAY RESULTS
# ============================================================
print(f"\nTest example row: {CONTEXT_ROW_INDEX:,}")
print(
    f"Visitor: {int(example_information['visitorid']):,}"
)
print("Expected next item(s):", actual_itemids)
print(f"Initially ranked: Top {NUMBER_OF_RESULTS:,}")
print(
    f"Show only true predictions: "
    f"{SHOW_ONLY_TRUE_PREDICTIONS}"
)
print(
    f"Rows displayed after filtering: "
    f"{len(ranked_prediction_output):,}"
)

print("\nSummary CSV saved to:")
print(summary_csv_path)

print("\nRanked predictions CSV saved to:")
print(ranked_csv_path)

display(clean_prediction_output)
display(ranked_prediction_output)

Showing only correct recommendations found inside the selected Top-N results.
No actual next item appeared in the selected Top-50 recommendations.

Test example row: 0
Visitor: 0
Expected next item(s): [357564]
Initially ranked: Top 50
Show only true predictions: True
Rows displayed after filtering: 0

Summary CSV saved to:
E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\test_outputs\test_summary_20261014_194055.csv

Ranked predictions CSV saved to:
E:\ALL MCP Servers\Recommendation\models\sasrec_basket_v1\test_outputs\test_ranked_predictions_20261014_194055.csv


,context_row_index,visitorid,target_timestamp,expected_next_itemids,predicted_next_itemid,predicted_logit_score,top_1_prediction_correct
0,0,0,1442004759591,[357564],285930,20.129459,False


,rank,predicted_item_index,predicted_itemid,logit_score,is_actual_next_item


In [30]:
# Exact rank of each actual next item among all 200,978 items.
actual_rank_rows = []

for actual_item_index, actual_itemid in zip(
    actual_item_indices,
    actual_itemids,
):
    actual_score = full_scores[
        actual_item_index - 1
    ]

    # Rank 1 means the highest-scoring item.
    actual_full_rank = int(
        (full_scores > actual_score).sum().item() + 1
    )

    actual_rank_rows.append(
        {
            "actual_item_index": actual_item_index,
            "actual_itemid": actual_itemid,
            "actual_logit_score": round(
                float(actual_score),
                6,
            ),
            "full_catalogue_rank": actual_full_rank,
            "inside_top_10": actual_full_rank <= 10,
            "inside_top_50": actual_full_rank <= 50,
        }
    )

actual_rank_output = pd.DataFrame(actual_rank_rows)

display(actual_rank_output)

,actual_item_index,actual_itemid,actual_logit_score,full_catalogue_rank,inside_top_10,inside_top_50
0,154167,357564,0.046636,24906,False,False


In [31]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ============================================================
# CONTROLLERS
# ============================================================
FULL_CATALOGUE_BATCH_SIZE = 256
SAVE_TOP_50_ITEM_IDS = False
# True makes the CSV larger, but stores all recommended IDs.
# ============================================================

PROJECT_ROOT = Path(
    r"E:\ALL MCP Servers\Recommendation"
)

MODEL_DIR = PROJECT_ROOT / "models" / "sasrec_basket_v1"

REFRAME_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "chronological_split_70_15_15"
    / "training_reframe"
)

CONTEXT_VECTOR_FILE = (
    MODEL_DIR / "final_sasrec_test_context_vectors.npy"
)

CONTEXT_METADATA_FILE = (
    MODEL_DIR / "final_sasrec_test_context_metadata.csv"
)

FINAL_SASREC_PATH = (
    MODEL_DIR / "best_model_by_ndcg10.pt"
)

FULL_RESULTS_FILE = (
    MODEL_DIR / "full_catalogue_test_per_example.csv"
)

FULL_METRICS_FILE = (
    MODEL_DIR / "full_catalogue_test_metrics.csv"
)

# The model architecture must already exist from an earlier cell.
assert "sasrec_model" in globals(), (
    "Run the SASRec model-definition cell first."
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "CUDA GPU was not detected."

# Load the chosen final model checkpoint.
checkpoint = torch.load(
    FINAL_SASREC_PATH,
    map_location=device,
    weights_only=False,
)

sasrec_model.load_state_dict(
    checkpoint["model_state_dict"]
)

sasrec_model.to(device)
sasrec_model.eval()

# Load all compact test contexts without loading the entire file into RAM.
saved_contexts = np.load(
    CONTEXT_VECTOR_FILE,
    mmap_mode="r",
)

context_metadata = pd.read_csv(
    CONTEXT_METADATA_FILE
)

assert len(saved_contexts) == len(context_metadata)

# Load model index -> original Retailrocket item ID mapping.
item_mapping = pd.read_csv(
    REFRAME_DIR / "train_item_id_mapping.csv"
)

NUM_TRAIN_ITEMS = int(
    item_mapping["item_index"].max()
)

index_to_itemid = np.zeros(
    NUM_TRAIN_ITEMS + 1,
    dtype=np.int64,
)

index_to_itemid[
    item_mapping["item_index"].to_numpy(dtype=np.int64)
] = item_mapping["itemid"].to_numpy(dtype=np.int64)

assert (index_to_itemid[1:] != 0).all()

# Convert true target lists once.
actual_item_index_lists = [
    json.loads(value)
    for value in context_metadata["target_item_indices"]
]

MAX_TARGET_ITEMS = max(
    len(item_list)
    for item_list in actual_item_index_lists
)

# Make a padded target matrix:
# 0 means padding; real model item indices start at 1.
target_index_matrix = np.zeros(
    (
        len(context_metadata),
        MAX_TARGET_ITEMS,
    ),
    dtype=np.int64,
)

target_item_counts = np.zeros(
    len(context_metadata),
    dtype=np.int64,
)

for row_number, item_list in enumerate(
    actual_item_index_lists
):
    target_index_matrix[
        row_number,
        :len(item_list),
    ] = item_list

    target_item_counts[row_number] = len(item_list)

assert (target_item_counts >= 1).all()

# All item vectors already belong to the loaded model.
# Padding vector is excluded, so row 0 represents item_index 1.
catalogue_item_vectors = (
    sasrec_model.basket_encoder.item_embedding.weight[1:]
).float()

assert catalogue_item_vectors.shape[0] == NUM_TRAIN_ITEMS

# We will report these exact full-catalogue ranking measures.
K_VALUES = [1, 10, 20, 50]
MAX_K = max(K_VALUES)

discounts = 1.0 / torch.log2(
    torch.arange(
        2,
        MAX_K + 2,
        device=device,
        dtype=torch.float32,
    )
)

metric_totals = {
    k: {
        "hit_count": 0,
        "recall_sum": 0.0,
        "ndcg_sum": 0.0,
    }
    for k in K_VALUES
}

# Remove an old output so this run starts cleanly.
if FULL_RESULTS_FILE.exists():
    FULL_RESULTS_FILE.unlink()

start_time = time.perf_counter()
rows_written = 0

print(
    f"Starting full-catalogue evaluation for "
    f"{len(context_metadata):,} test examples."
)
print(
    f"Each example is compared with "
    f"{NUM_TRAIN_ITEMS:,} known training items."
)

with torch.inference_mode():
    for batch_start in range(
        0,
        len(context_metadata),
        FULL_CATALOGUE_BATCH_SIZE,
    ):
        batch_end = min(
            batch_start + FULL_CATALOGUE_BATCH_SIZE,
            len(context_metadata),
        )

        batch_size = batch_end - batch_start

        # .copy() prevents the read-only NumPy warning.
        context_batch = torch.from_numpy(
            np.asarray(
                saved_contexts[batch_start:batch_end]
            ).copy()
        ).to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        target_index_batch = torch.from_numpy(
            target_index_matrix[batch_start:batch_end]
        ).to(
            device,
            dtype=torch.long,
            non_blocking=True,
        )

        target_count_batch = torch.from_numpy(
            target_item_counts[batch_start:batch_end]
        ).to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        target_mask = target_index_batch.ne(0)

        # Full score matrix:
        # [batch_size, 64] x [64, 200978]
        #
        # float32 is used here so ranks are consistent and auditable.
        score_matrix = torch.matmul(
            context_batch,
            catalogue_item_vectors.T,
        ) * (catalogue_item_vectors.size(1) ** -0.5)

        # Top 50 recommendations.
        top_scores, top_positions = torch.topk(
            score_matrix,
            k=MAX_K,
            dim=1,
        )

        # Convert zero-based score positions to model item indices.
        top_item_indices = top_positions + 1

        # Does each recommended item match any actual next-basket item?
        relevance_top_50 = (
            top_item_indices.unsqueeze(2)
            .eq(target_index_batch.unsqueeze(1))
            .any(dim=2)
        )

        # Find the highest-scoring actual target item.
        safe_target_positions = (
            target_index_batch.clamp_min(1) - 1
        )

        actual_target_scores = score_matrix.gather(
            1,
            safe_target_positions,
        )

        actual_target_scores = actual_target_scores.masked_fill(
            ~target_mask,
            float("-inf"),
        )

        best_actual_scores, best_actual_columns = (
            actual_target_scores.max(dim=1)
        )

        best_actual_item_indices = target_index_batch.gather(
            1,
            best_actual_columns.unsqueeze(1),
        ).squeeze(1)

        # Exact minimum rank among the actual next items.
        # Rank 1 is best.
        best_actual_full_ranks = (
            score_matrix
            .gt(best_actual_scores.unsqueeze(1))
            .sum(dim=1)
            + 1
        )

        # Aggregate exact metrics.
        for k in K_VALUES:
            relevance_at_k = relevance_top_50[:, :k]

            hit_at_k = relevance_at_k.any(dim=1)

            recall_at_k = (
                relevance_at_k.sum(dim=1).float()
                / target_count_batch
            )

            dcg_at_k = (
                relevance_at_k.float()
                * discounts[:k]
            ).sum(dim=1)

            ideal_item_counts = target_count_batch.long().clamp(
                max=k
            )

            idcg_at_k = torch.zeros_like(dcg_at_k)

            for target_count in range(1, MAX_TARGET_ITEMS + 1):
                matching_rows = ideal_item_counts.eq(
                    min(target_count, k)
                )

                if matching_rows.any():
                    idcg_at_k[matching_rows] = discounts[
                        :min(target_count, k)
                    ].sum()

            ndcg_at_k = dcg_at_k / idcg_at_k

            metric_totals[k]["hit_count"] += int(
                hit_at_k.sum().item()
            )

            metric_totals[k]["recall_sum"] += float(
                recall_at_k.sum().item()
            )

            metric_totals[k]["ndcg_sum"] += float(
                ndcg_at_k.sum().item()
            )

        # Move only compact output data back to CPU.
        top_item_indices_np = (
            top_item_indices.cpu().numpy()
        )

        top_scores_np = top_scores.cpu().numpy()

        relevance_top_50_np = (
            relevance_top_50.cpu().numpy()
        )

        best_actual_item_indices_np = (
            best_actual_item_indices.cpu().numpy()
        )

        best_actual_scores_np = (
            best_actual_scores.cpu().numpy()
        )

        best_actual_full_ranks_np = (
            best_actual_full_ranks.cpu().numpy()
        )

        batch_metadata = context_metadata.iloc[
            batch_start:batch_end
        ].copy()

        output_rows = []

        for local_row, (_, metadata_row) in enumerate(
            batch_metadata.iterrows()
        ):
            actual_indices = actual_item_index_lists[
                batch_start + local_row
            ]

            actual_itemids = [
                int(index_to_itemid[item_index])
                for item_index in actual_indices
            ]

            top_50_indices = top_item_indices_np[
                local_row
            ]

            top_50_itemids = index_to_itemid[
                top_50_indices
            ]

            retrieved_actual_itemids = [
                int(itemid)
                for itemid, is_relevant in zip(
                    top_50_itemids,
                    relevance_top_50_np[local_row],
                )
                if is_relevant
            ]

            top_1_is_correct = bool(
                relevance_top_50_np[local_row, 0]
            )

            hit_at_10 = bool(
                relevance_top_50_np[local_row, :10].any()
            )

            hit_at_20 = bool(
                relevance_top_50_np[local_row, :20].any()
            )

            hit_at_50 = bool(
                relevance_top_50_np[local_row].any()
            )

            result_row = {
                "context_row_index": int(
                    metadata_row["context_row_index"]
                ),
                "visitorid": int(metadata_row["visitorid"]),
                "target_timestamp": int(
                    metadata_row["target_timestamp"]
                ),
                "target_block_id": metadata_row[
                    "target_block_id"
                ],
                "history_length": int(
                    metadata_row["history_length"]
                ),
                "actual_next_itemids": json.dumps(
                    actual_itemids
                ),
                "actual_item_count": len(actual_itemids),
                "predicted_top_1_itemid": int(
                    top_50_itemids[0]
                ),
                "predicted_top_1_logit_score": round(
                    float(top_scores_np[local_row, 0]),
                    6,
                ),
                # TP means the rank-1 recommendation is correct.
                "top_1_result": (
                    "TP"
                    if top_1_is_correct
                    else "MISS"
                ),
                # HIT means at least one actual item appears in Top 50.
                "top_50_result": (
                    "TP_IN_TOP_50"
                    if hit_at_50
                    else "FAILED_TOP_50"
                ),
                "hit_at_10": hit_at_10,
                "hit_at_20": hit_at_20,
                "hit_at_50": hit_at_50,
                "best_actual_itemid": int(
                    index_to_itemid[
                        best_actual_item_indices_np[local_row]
                    ]
                ),
                "best_actual_logit_score": round(
                    float(
                        best_actual_scores_np[local_row]
                    ),
                    6,
                ),
                "best_actual_full_catalogue_rank": int(
                    best_actual_full_ranks_np[local_row]
                ),
                "retrieved_actual_itemids_in_top_50": (
                    json.dumps(retrieved_actual_itemids)
                ),
                "retrieved_actual_item_count_in_top_50": (
                    len(retrieved_actual_itemids)
                ),
            }

            if SAVE_TOP_50_ITEM_IDS:
                result_row["top_50_predicted_itemids"] = (
                    json.dumps(
                        [
                            int(itemid)
                            for itemid in top_50_itemids
                        ]
                    )
                )

            output_rows.append(result_row)

        output_frame = pd.DataFrame(output_rows)

        output_frame.to_csv(
            FULL_RESULTS_FILE,
            mode=(
                "w"
                if batch_start == 0
                else "a"
            ),
            header=(batch_start == 0),
            index=False,
        )

        rows_written += len(output_frame)

        if (
            rows_written % (
                FULL_CATALOGUE_BATCH_SIZE * 20
            ) == 0
            or rows_written == len(context_metadata)
        ):
            elapsed_minutes = (
                time.perf_counter() - start_time
            ) / 60

            print(
                f"Processed {rows_written:,}/"
                f"{len(context_metadata):,} examples "
                f"({elapsed_minutes:.1f} minutes)"
            )

        del score_matrix

assert rows_written == len(context_metadata)

elapsed_seconds = time.perf_counter() - start_time

# Create one final metrics table.
metric_rows = []

for k in K_VALUES:
    metric_rows.append(
        {
            "evaluation_protocol": (
                "full catalogue: all training-known "
                "items scored for every test example"
            ),
            "checkpoint_epoch": int(checkpoint["epoch"]),
            "test_examples": len(context_metadata),
            "catalogue_items": NUM_TRAIN_ITEMS,
            "k": k,
            "event_hit_rate": (
                metric_totals[k]["hit_count"]
                / len(context_metadata)
            ),
            "multi_positive_recall": (
                metric_totals[k]["recall_sum"]
                / len(context_metadata)
            ),
            "ndcg": (
                metric_totals[k]["ndcg_sum"]
                / len(context_metadata)
            ),
            "elapsed_minutes": round(
                elapsed_seconds / 60,
                2,
            ),
        }
    )

full_catalogue_metrics = pd.DataFrame(metric_rows)

full_catalogue_metrics.to_csv(
    FULL_METRICS_FILE,
    index=False,
)

print("\n✅ Full-catalogue evaluation completed.")
print("Per-example audit CSV:", FULL_RESULTS_FILE)
print("Exact metrics CSV:", FULL_METRICS_FILE)

display(full_catalogue_metrics)

# Examples of successes and failures.
full_results_preview = pd.read_csv(
    FULL_RESULTS_FILE,
    nrows=10,
)

display(full_results_preview)

Starting full-catalogue evaluation for 173,786 test examples.
Each example is compared with 200,978 known training items.
Processed 5,120/173,786 examples (0.0 minutes)
Processed 10,240/173,786 examples (0.0 minutes)
Processed 15,360/173,786 examples (0.0 minutes)
Processed 20,480/173,786 examples (0.0 minutes)
Processed 25,600/173,786 examples (0.0 minutes)
Processed 30,720/173,786 examples (0.1 minutes)
Processed 35,840/173,786 examples (0.1 minutes)
Processed 40,960/173,786 examples (0.1 minutes)
Processed 46,080/173,786 examples (0.1 minutes)
Processed 51,200/173,786 examples (0.1 minutes)
Processed 56,320/173,786 examples (0.1 minutes)
Processed 61,440/173,786 examples (0.1 minutes)
Processed 66,560/173,786 examples (0.1 minutes)
Processed 71,680/173,786 examples (0.1 minutes)
Processed 76,800/173,786 examples (0.1 minutes)
Processed 81,920/173,786 examples (0.1 minutes)
Processed 87,040/173,786 examples (0.2 minutes)
Processed 92,160/173,786 examples (0.2 minutes)
Processed 97,28

,evaluation_protocol,checkpoint_epoch,test_examples,catalogue_items,k,event_hit_rate,multi_positive_recall,ndcg,elapsed_minutes
0,full catalogue: all training-known items score...,15,173786,200978,1,0.316280,0.315964,0.316280,0.31
1,full catalogue: all training-known items score...,15,173786,200978,10,0.448241,0.447943,0.381140,0.31
2,full catalogue: all training-known items score...,15,173786,200978,20,0.477259,0.476988,0.388492,0.31
3,full catalogue: all training-known items score...,15,173786,200978,50,0.514880,0.514586,0.395982,0.31


,context_row_index,visitorid,target_timestamp,target_block_id,history_length,actual_next_itemids,actual_item_count,predicted_top_1_itemid,predicted_top_1_logit_score,top_1_result,top_50_result,hit_at_10,hit_at_20,hit_at_50,best_actual_itemid,best_actual_logit_score,best_actual_full_catalogue_rank,retrieved_actual_itemids_in_top_50,retrieved_actual_item_count_in_top_50
0,0,0,1442004759591,v0_t1442004759591,1,[357564],1,285930,20.127016,MISS,FAILED_TOP_50,False,False,False,357564,0.046914,24907,[],0
1,1,0,1442004917175,v0_t1442004917175,2,[67045],1,285930,13.110263,MISS,TP_IN_TOP_50,False,False,True,67045,8.544577,23,[67045],1
2,2,6,1440916778318,v6_t1440916778318,1,[253615],1,65273,16.767431,MISS,TP_IN_TOP_50,True,True,True,253615,10.182816,7,[253615],1
3,3,6,1440916823805,v6_t1440916823805,2,[344723],1,253615,15.713696,MISS,TP_IN_TOP_50,True,True,True,344723,11.607075,5,[344723],1
4,4,6,1440917649385,v6_t1440917649385,3,[344723],1,344723,16.118189,TP,TP_IN_TOP_50,True,True,True,344723,16.118189,1,[344723],1
5,5,6,1440917793838,v6_t1440917793838,4,[344723],1,344723,16.734150,TP,TP_IN_TOP_50,True,True,True,344723,16.734150,1,[344723],1
6,6,6,1440991285697,v6_t1440991285697,5,[344723],1,344723,17.102455,TP,TP_IN_TOP_50,True,True,True,344723,17.102455,1,[344723],1
7,7,32,1441905247409,v32_t1441905247409,1,[282491],1,465465,17.047010,MISS,FAILED_TOP_50,False,False,False,282491,-1.984097,128653,[],0
8,8,51,1441495416363,v51_t1441495416363,1,[429304],1,198762,17.041355,MISS,TP_IN_TOP_50,True,True,True,429304,7.324356,9,[429304],1
9,9,51,1441495582729,v51_t1441495582729,2,[198762],1,429304,16.059414,MISS,TP_IN_TOP_50,True,True,True,198762,14.558739,2,[198762],1
